# NER Extraction + Entity-Aware QG for contraTICO

This notebook runs:
1. **NER Extraction** - Extract entities from the 84 unique source sentences
2. **Entity-Aware QG** - Generate entity-specific questions using Qwen

## Environment Setup

In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
elif IN_KAGGLE:
    print('Running on Kaggle')
else:
    print('Running locally')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'torch', 'accelerate', 'nltk',
                'sentence-transformers', 'sacrebleu', 'textstat'], check=True)
print('Dependencies installed!')

In [ ]:
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()

print(f'Project root: {PROJECT_ROOT}')

## Path Configuration

In [ ]:
RESULTS_DIR = f"{PROJECT_ROOT}/results Qwen3B baseline"
EXTENSION_DIR = f"{RESULTS_DIR}/contratico/ner-extension"
CONTRATICO_CODE_DIR = f"{EXTENSION_DIR}/code"
BIOMQM_NER_CODE_DIR = f"{RESULTS_DIR}/biomqm/ner-extension/code"

# contraTICO original data (for reading 'en' sentences)
CONTRATICO_DATA_DIR = f"{PROJECT_ROOT}/contratico"
# Use any file - they all share the same 'en' sentences
CONTRATICO_SOURCE_FILE = f"{CONTRATICO_DATA_DIR}/en-es/alteration.jsonl"

# Outputs
NER_OUTPUT = f"{EXTENSION_DIR}/QG/ner_output.jsonl"
QG_OUTPUT = f"{EXTENSION_DIR}/QG/qg_entity_aware.jsonl"

# Sampling config (must match subset creation)
SAMPLE_SIZE = 84
SEED = 42

# Create directories
os.makedirs(f"{EXTENSION_DIR}/QG", exist_ok=True)
os.makedirs(f"{EXTENSION_DIR}/QA", exist_ok=True)

# Add code directories to path
for d in [CONTRATICO_CODE_DIR, BIOMQM_NER_CODE_DIR]:
    if d not in sys.path:
        sys.path.insert(0, d)

print(f"EXTENSION_DIR: {EXTENSION_DIR}")
print(f"CONTRATICO_SOURCE_FILE: {CONTRATICO_SOURCE_FILE}")

## Step 1: NER Extraction

Extract entities from 84 unique English source sentences.

In [ ]:
cmd = [
    sys.executable, "-u",
    f"{CONTRATICO_CODE_DIR}/ner_extraction_contratico.py",
    "--input_path", CONTRATICO_SOURCE_FILE,
    "--output_path", NER_OUTPUT,
    "--sample_size", str(SAMPLE_SIZE),
    "--seed", str(SEED)
]

print("Running NER extraction...")
subprocess.run(cmd, check=True)
print("✓ NER Extraction complete!")

## Step 2: Entity-Aware Question Generation

Generate entity-specific questions using Qwen.
Reuses `qg_entity_aware.py` from biomqm NER extension.

In [ ]:
cmd = [
    sys.executable, "-u",
    f"{BIOMQM_NER_CODE_DIR}/qg_entity_aware.py",
    "--input_path", NER_OUTPUT,
    "--output_path", QG_OUTPUT
]

print("Running Entity-Aware QG...")
subprocess.run(cmd, check=True)
print("✓ QG complete!")

## Verification

In [ ]:
import json

for label, path in [("NER", NER_OUTPUT), ("QG", QG_OUTPUT)]:
    if os.path.exists(path):
        with open(path) as f:
            rows = [json.loads(l) for l in f]
        print(f"✓ {label}: {len(rows)} rows")
        if rows:
            print(f"  Keys: {list(rows[0].keys())}")
            if 'entities' in rows[0]:
                total_ent = sum(len(r.get('entities', [])) for r in rows)
                print(f"  Total entities: {total_ent}")
            if 'questions' in rows[0]:
                total_q = sum(len(r.get('questions', [])) for r in rows)
                print(f"  Total questions: {total_q}")
    else:
        print(f"✗ {label}: NOT FOUND")